# Introduction to Neural Network Architectures

- [View Solution Notebook](./solutions.html)
- [View Project Page](https://www.codecademy.com/content-items/c3e07b9eb8e5444b8b466ab95732c3c0)
  
In this project, you apply your knowledge of neural network architectures by building and training various models for two tasks using popular real-world datasets:

## Objectives

**A.** Build and compare a simple MLP and a deep CNN on the classic MNIST handwritten digit classification problem.
- Load and preprocess the [MNIST dataset](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.MNIST.html)
- Build an MLP architecture
- Build a ResNet18 architecture
- Profile, evaluate, and compare their performance

**B.** Build and compare an RNN, GRU, and LSTM to predict hourly household power usage from historical measurements.
- Load the [Household Electric Power Consumption dataset](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption)
- Construct and preprocess sequences for model input
- Build and train RNN, GRU, and LSTM models
- Profile, evaluate, and compare their performance

### Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as M
from sklearn.metrics import accuracy_score, classification_report
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Objective A: Predict Handwritten Images

## Task Group 1: MNIST Data Loading and Preprocessing

**Introduction to the MNIST Dataset**

The [MNIST dataset](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.MNIST.html) is one of the most popular and widely used benchmark datasets for image classification in machine learning. It consists of **70,000 images** of handwritten digits (0–9), each sized **28 × 28 pixels**, with a single grayscale channel. By default, 60,000 images are used for training, and 10,000 images are used for testing.

MNIST is commonly used for:
- Practicing image preprocessing, model training, and evaluation workflows.
- Building and evaluating image classification models.  
- Benchmarking neural network architectures such as MLPs and CNNs.  

Below is one sample image from each class (0–9):

![MNIST sample digits](mnist_samples.jpg)

### Task 1: Define Image Transformations
Define transformations to:
- Resize images to 28x28 (same as original)
- Convert to tensors
- Normalize the single channel, respectively, with mean `[0.1307]` and standard deviation `[0.3081]`

In [2]:
transform = transforms.Compose([
    transforms.Resize(28),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307],
                         std=[0.3081]),
])

### Task 2: Load MNIST Datasets
Load both the MNIST training and test datasets with the transforms you defined into the `datasets` directory.

The MNIST datasets can be located using `datasets.MNIST` from the `torchvision.datasets` module.

In [3]:
from pathlib import Path

data_dir = "./datasets"

train_dataset  = datasets.MNIST(root=data_dir, train=True, download=False, transform=transform)
test_dataset  = datasets.MNIST(root=data_dir, train=False, download=False, transform=transform)

### Task 3: Create DataLoaders
Create DataLoader objects with:
- Training: batch size of `32` and set `shuffle=True`.
- Testing: batch size of `32` and set `shuffle=False`.

In [4]:
train_loader  = DataLoader(
        train_dataset,  
        batch_size=32, 
        shuffle=True
    )

print(f"Loaded {len(train_dataset):,} MNIST train images from {Path(data_dir).resolve()}")

test_loader  = DataLoader(
        test_dataset,  
        batch_size=32, 
        shuffle=False
    )

print(f"Loaded {len(test_dataset):,} MNIST test images from {Path(data_dir).resolve()}")

Loaded 60,000 MNIST train images from C:\Users\nomadant\AI_Engineering_Lessons\Codecademy\Neural Network Architectures\nn-architectures-project1\datasets
Loaded 10,000 MNIST test images from C:\Users\nomadant\AI_Engineering_Lessons\Codecademy\Neural Network Architectures\nn-architectures-project1\datasets


## Task Group 2: Multi-Layer Perceptron (MLP) and ResNet18 Architectures

### Task 4: Define SimpleMLP Class
Create a **simple MLP** with the following default values:
- Input size: 28x28x1 (flattened image with one channel)
- Hidden size: 128
- Output size: 10 (digit classes)
- 3 fully connected layers with ReLU activations after the first and second fully connected layers

In [5]:
class SimpleMLP(nn.Module):
    def __init__(self, input_size=28*28, hidden_size=128, output_size=10):
        super().__init__()
        self.fc1   = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Linear(hidden_size, hidden_size)
        self.relu2 = nn.ReLU()
        self.fc3   = nn.Linear(hidden_size, output_size) 

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten the input
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

### Task 5: Instantiate MLP Model
Create an instance of **SimpleMLP** class and move it to the device.  Be sure to correctly specify the default input size, hidden size, and output size. 

Profile the model size using an input size with the format: `(batch_size, channels, width, height)`

In [6]:
model_mlp = SimpleMLP().to(device)
model_mlp.eval()

from custom_torchinfo import custom_summary
custom_summary(model_mlp, input_size=(32, 1, 28, 28))

        Layer (type)              Output Shape         Param #
            Linear-1                 [32, 128]         100,480
              ReLU-2                 [32, 128]               0
            Linear-3                 [32, 128]          16,512
              ReLU-4                 [32, 128]               0
            Linear-5                  [32, 10]           1,290
         SimpleMLP-6                  [32, 10]               0
Total params: 118,282
Trainable params: 118,282
Non-trainable params: 0


### Task 6: Load and Modify ResNet18
Load **ResNet18** without pretrained weights and modify the final layer for 10 classes. Be sure to place the model to the device.

We'll need to modify the convolutional layer to accept a single channel (from three) using:

```py
model_resnet18.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
model_resnet18.maxpool = nn.Identity() # removes maxpool layer
```

Profile the model size using an input size with the format: `(batch_size, channels, width, height)`

In [7]:
model_resnet18 = M.resnet18(weights=None)
in_feats = model_resnet18.fc.in_features
model_resnet18.fc = nn.Linear(in_feats, 10)
model_resnet18.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
model_resnet18.maxpool = nn.Identity() # type: ignore[assignment], removes maxpool layer

# Move to device and set to evaluation mode
model_resnet18 = model_resnet18.to(device)
model_resnet18.eval()

# Show output profile
from custom_torchinfo import custom_summary
custom_summary(model_resnet18, input_size=(32, 1, 28, 28))

        Layer (type)              Output Shape         Param #
            Conv2d-1          [32, 64, 28, 28]             576
       BatchNorm2d-2          [32, 64, 28, 28]             128
              ReLU-3          [32, 64, 28, 28]               0
          Identity-4          [32, 64, 28, 28]               0
            Conv2d-5          [32, 64, 28, 28]          36,864
       BatchNorm2d-6          [32, 64, 28, 28]             128
              ReLU-7          [32, 64, 28, 28]               0
            Conv2d-8          [32, 64, 28, 28]          36,864
       BatchNorm2d-9          [32, 64, 28, 28]             128
             ReLU-10          [32, 64, 28, 28]               0
       BasicBlock-11          [32, 64, 28, 28]               0
           Conv2d-12          [32, 64, 28, 28]          36,864
      BatchNorm2d-13          [32, 64, 28, 28]             128
             ReLU-14          [32, 64, 28, 28]               0
           Conv2d-15          [32, 64, 28, 28]         

## Task Group 3: Train and Evaluate the Vision Models

### Task 7: Define Training Function

Create a function that:
- Initializes the Cross-Entropy loss function, Adam optimizer, and sets the model to training mode.
- Trains a model for multiple epochs, while keeping track of the training loss and accuracy.

In [8]:
def train_model(model, train_loader, epochs, learning_rate):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    model.train()

    for epoch in range(epochs):
        epoch_loss = 0
        correct, total = 0, 0
        num_batches = 0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Forward pass
            logits = model(batch_X)

            # Compute loss
            loss = loss_fn(logits, batch_y)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Track loss
            epoch_loss += loss.item()
            num_batches += 1

            # Convert logits to predicted labels
            preds = torch.argmax(logits, dim=1)

            # Accuracy
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
    
        # Average metrics
        avg_loss = epoch_loss / num_batches
        accuracy = correct / total

        # Print training progress
        print(f'Epoch [{epoch+1}/{epochs}], CE Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')

### Task 8: Train ResNet18 Model
Train the ResNet18 model for `1` epoch with a learning rate of `0.001`.

In [9]:
train_model(model_resnet18, train_loader, epochs=1, learning_rate=0.001)

Epoch [1/1], CE Loss: 0.1114, Accuracy: 0.9662


### Task 9: Train MLP Model
Train the MLP model for `3` epochs with a learning rate of `0.001`.

In [10]:
train_model(model_mlp, train_loader, epochs=3, learning_rate=0.001)

Epoch [1/3], CE Loss: 0.2256, Accuracy: 0.9320
Epoch [2/3], CE Loss: 0.1008, Accuracy: 0.9689
Epoch [3/3], CE Loss: 0.0739, Accuracy: 0.9762


### Task 10: Define Prediction Function
Create a function that builds the prediction loop to generate predictions on the testing set images.

In [11]:
def predict(model, dataloader, device="cpu"):
    model.eval()
    all_logits = []
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            logits = model(batch_X)
            preds = torch.argmax(logits, dim=1)

            all_logits.append(logits.cpu().numpy())
            all_predictions.append(preds.cpu().numpy())
            all_labels.append(batch_y.cpu().numpy())

    y_logits = np.concatenate(all_logits, axis=0)
    y_pred   = np.concatenate(all_predictions, axis=0)
    y_true   = np.concatenate(all_labels, axis=0)

    return y_logits, y_pred, y_true

### Task 11: Generate Predictions
Generate predictions for both models on the test set.

In [12]:
y_logits_mlp, y_pred_mlp, y_true_mlp = predict(model_mlp, test_loader, device="cuda")
y_logits_resnet18, y_pred_resnet18, y_true_resnet18 = predict(model_resnet18, test_loader, device="cuda")

### Task 12: Compare Model Performance
Calculate and display accuracy and classification reports for both models.

In [13]:
accuracy_mlp = accuracy_score(y_true_mlp, y_pred_mlp)
report_mlp = classification_report(y_true_mlp, y_pred_mlp,
                                   target_names=[str(i) for i in range(10)],
                                   output_dict=True)

accuracy_resnet18 = accuracy_score(y_true_resnet18, y_pred_resnet18)
report_resnet18 = classification_report(y_true_resnet18, y_pred_resnet18,
                                        target_names=[str(i) for i in range(10)],
                                        output_dict=True)

# Show output - Compare Model Performances
from typing import Dict, Any, cast

report_mlp = cast(Dict[str, Any], report_mlp)
report_resnet18 = cast(Dict[str, Any], report_resnet18)

reports: Dict[str, Dict[str, Any]] = {
    "MLP": report_mlp,
    "ResNet18": report_resnet18
}

results = {
    model: {
        "Accuracy": report["accuracy"],
        "Macro Precision": report["macro avg"]["precision"],
        "Macro Recall": report["macro avg"]["recall"],
        "Macro F1": report["macro avg"]["f1-score"],
    }
    for model, report in reports.items()
}

compare_df = pd.DataFrame(results).T
compare_df.round(6)

,Accuracy,Macro Precision,Macro Recall,Macro F1
MLP,0.9711,0.971410,0.970761,0.970997
ResNet18,0.9870,0.987093,0.986880,0.986932


## Conclusion (Part A)

Compare the results of the MLP and ResNet18 models:
- Which model performed better?
- What are the advantages of CNNs over MLPs for image classification?
- How do the training times compare?
- Which digits were most difficult to classify for each model?

### Task (Optional): Clean Up

In [14]:
# Clean Up Part A
import gc, torch

# 1) Move models off GPU (safe no-op on CPU)
for m in [globals().get('model_mlp'), globals().get('model_resnet18')]:
    if m is not None:
        try:
            m.to('cpu')
        except Exception:
            pass

# 2) Delete unneeded items
for name in [
    'model_mlp', 'optimizer_mlp', 'scheduler_mlp', 'scaler_mlp', 'criterion_mlp',
    'y_pred_mlp', 'y_true',  
     'train_loader', 'test_loader',     
     'train_dataset', 'test_dataset',
]:
    globals().pop(name, None)

# 3) Collect + clear CUDA cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

## Objective B - Predict Hourly Household Power Usage

## Task Group 4: Household Power Data Loading and Preprocessing

**Introduction to the Household Power Dataset**

The [Household Electric Power Consumption dataset](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption) contains real-world one-minute measurements of one household's electric power consumption (in kilowatts) over four years. 

Key features include:
- `Global_active_power`: Total active power consumed (kilowatts)  
- `Global_reactive_power`: Reactive power (kilowatts)  
- `Voltage`: Average voltage (volts)  
- `Global_intensity`: Current intensity (amperes)  
- `Sub_metering_1`, `Sub_metering_2`, `Sub_metering_3`: Energy usage from different sub-circuits (watt-hours)
- `Timestamp`: Represents the date and time of each hourly observation

For this task, we'll use a smaller subset of the full dataset containing **hourly measurements** instead. Predicting hourly consumption instead of one-minute consumption may help reduce noise, shorten sequence lengths, and reduce overfitting while still preserving daily/weekly patterns.

**Note:** If you find that your kernel crashes or restarts, be sure to run the cleanup cell above from the previous task. It may also be useful to restart the notebook in the **Kernel > Restart Kernel** tab and begin coding from here. 


### Setup and Imports

In [15]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Task 13: Load the Training Dataset

Load the training dataset from `"datasets/household_power_train.csv"` into a pandas and set the `"Timestamp"` column as the DataFrame index.

### Task 14: Create Lagged Features Function
Create a function called `make_lag_df()` that takes a DataFrame, feature columns, and a window size as parameters. The function should create lagged features by shifting each feature column backward in time for the specified window size.

### Task 15: Apply Lagged Features
Apply the `make_lag_df()` function to create the lagged training DataFrame with a lagged window of 24 hours. 

Be sure to save the lagged features as a new list.

### Task 16: Create Target Variable
Create a target column that contains the next hour's `Global_active_power` values. Use the original DataFrame's values aligned by index and convert the target column to float values.

### Task 17: Create DataLoader Function
Define a function `df_to_loader()` that converts a sequential DataFrame into a PyTorch DataLoader. The function should reshape the flat lagged features into 3D tensors (batch, timesteps, features) and create a TensorDataset.

### Task 18: Create Training DataLoader
Create a training DataLoader by calling `df_to_loader()` with the sequential training data. Specify a batch size and be sure to shuffle the training data.

## Task Group 5: Build and Train Neural Network Models

### Task 19: Define RNN, GRU, and LSTM Models

**A.** Define a class for an RNN that inherits from `nn.Module`. Include an RNN layer with configurable input size, hidden size, and number of layers, followed by a fully connected layer that outputs a single value.

**B.** Define a class for a GRU that inherits from `nn.Module`. Include a GRU layer with the same configuration as SimpleRNN, followed by a fully connected output layer.

**C.** Define a class for an LSTM that inherits from `nn.Module`. Include an LSTM layer with the same configuration, followed by a fully connected output layer.

### Task 20: Instantiate Models
Instantiate all three model architectures (RNN, GRU, LSTM) with `input_size=7`, `hidden=64`, and `num_layers=2`. Move each model to the appropriate device.

### Task 21: Define Loss Function and Optimizers
Define the Mean Squared Error (MSE) loss function. For each model, create Adam optimizers with a learning rate of `0.001`.

### Task 22: Create Training Function
Create a reusable training function called `train_model()` that takes a model, optimizer, train_loader, number of epochs, and model name as parameters. The function should train the model and print the loss for each epoch.

### Task 23: Train the RNN, GRU, and LSTM Models
Use the `train_model()` function to separately train all three models (RNN, GRU, LSTM) for `10` epochs each.

## Task Group 6: Generate RNN, GRU, and LSTM Predictions

### Task 24: Define Prediction Function
Define a `predict()` function that creates the prediction loop for each RNN. The function should take a model and DataLoader, sets the model to evaluation mode, and generates predictions for all batches. The function should return the predictions across all batches.

### Task 25: Prepare Test Data
Load the test data from `"datasets/household_power_test.csv"`, create lagged features, add the target column, and make the test DataLoader without shuffling. 

### Task 26: Generate Model Predictions
Generate predictions for all three models on the test set using the `predict()` function.

### Task 27: Calculate MSE
Calculate the Mean Squared Error (MSE) for each model's predictions and print the results in a formatted comparison.

### Task 28: Display Next-Hour Predictions
Extract the second-to-last prediction from each model that predicts the hourly power consumption of the last observation in the test set. 

Display the predictions alongside the hour's actual power consumption of the last observation.

### Task 29: Calculate Model Size

Create a function named `model_size_bytes` that returns a PyTorch model's size in bytes (parameters + buffers). Apply the function to obtain the size of the RNN, GRU, and LSTM. 

## Conclusion (Part B)

Compare the results of the RNN, GRU, and LSTM:
- Which model produced the lowest MSE on the test set? What about the highest?
- Which model predicted the last hourly power usage the closest to the actual value?
- Was there a noticeable relationship between performance and model size?
- How could we adjust the training process to improve the performance? Think about how the training data was preprocessed into sequences.